<a href="https://colab.research.google.com/github/apirakqqqqq/GE337_Programming/blob/main/Lab_4/Lab_4_6606614870_%E0%B8%AD%E0%B8%A0%E0%B8%B4%E0%B8%A3%E0%B8%B1%E0%B8%81%E0%B8%A9%E0%B9%8C_%E0%B8%9B%E0%B8%B1%E0%B8%8D%E0%B8%8D%E0%B8%B2%E0%B8%AA%E0%B8%B2%E0%B8%84%E0%B8%A3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
!pip install geopandas rasterio folium shapely matplotlib -q

In [18]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**โหลดข้อมูล**

In [45]:
import rasterio

raster_path = "/content/S2_BKK_7Bands.tif"

src = rasterio.open(raster_path)

print("Metadata")
print(src.meta)

print("Number of bands:", src.count)
print("CRS:", src.crs)
print("Resolution:", src.res)

Metadata
{'driver': 'GTiff', 'dtype': 'float32', 'nodata': None, 'width': 3340, 'height': 3898, 'count': 7, 'crs': CRS.from_wkt('GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]'), 'transform': Affine(0.0001796630568239043, 0.0, 100.29995574391057,
       0.0, -0.0001796630568239043, 14.100316025653656)}
Number of bands: 7
CRS: EPSG:4326
Resolution: (0.0001796630568239043, 0.0001796630568239043)


In [46]:
import geopandas as gpd

vector_path = "/content/พื้นที่สีเขียว_เขตเมือง_แหล่งน้ำ_กรุงเทพ.json"

gdf = gpd.read_file(vector_path)

print(gdf.head())
print("Total points:", len(gdf))

   OBJECTID          F_id      landuse                               name  \
0         1  way/25776327   industrial  ศูนย์ซ่อมบำรุงรถไฟฟ้าสายสีน้ำเงิน   
1         2  way/27239594   industrial                               None   
2         3  way/27302885  residential                   แฟลตการเคหะบางนา   
3         4  way/29976632  residential                          บ้านสวนธน   
4         5  way/30135060   commercial              The Sukhothai Bangkok   

  natural water F_geometry  id_park park_name location   tel open_close  \
0    None  None       None      NaN      None     None  None       None   
1    None  None       None      NaN      None     None  None       None   
2    None  None       None      NaN      None     None  None       None   
3    None  None       None      NaN      None     None  None       None   
4    None  None       None      NaN      None     None  None       None   

   dcode dname  serviceuse  area  lat  lng                    geometry  
0    NaN  Non

สร้าง Class

In [47]:
def create_class(row):

    if row["natural"] == "water":
        return "water"

    elif row["landuse"] in ["industrial","residential","commercial"]:
        return "urban"

    elif pd.notna(row["park_name"]):
        return "green"

    else:
        return None


gdf["class"] = gdf.apply(create_class, axis=1)

In [48]:
import numpy as np

blue = src.read(1)
green = src.read(2)
red = src.read(3)
nir = src.read(4)

ndvi = (nir - red) / (nir + red + 1e-10)

**รวม Raster + Vector เป็น DataFrame**

In [49]:
coords = [(x,y) for x,y in zip(gdf.geometry.x, gdf.geometry.y)]

samples = []

for val in src.sample(coords):
    samples.append(val)

import pandas as pd

features = pd.DataFrame(samples, columns=[
    "Blue","Green","Red","NIR","SWIR1","SWIR2","Band7"
])

df = pd.concat([gdf.reset_index(drop=True), features], axis=1)

print(df.head())

   OBJECTID          F_id      landuse                               name  \
0         1  way/25776327   industrial  ศูนย์ซ่อมบำรุงรถไฟฟ้าสายสีน้ำเงิน   
1         2  way/27239594   industrial                               None   
2         3  way/27302885  residential                   แฟลตการเคหะบางนา   
3         4  way/29976632  residential                          บ้านสวนธน   
4         5  way/30135060   commercial              The Sukhothai Bangkok   

  natural water F_geometry  id_park park_name location  ... lng  \
0    None  None       None      NaN      None     None  ... NaN   
1    None  None       None      NaN      None     None  ... NaN   
2    None  None       None      NaN      None     None  ... NaN   
3    None  None       None      NaN      None     None  ... NaN   
4    None  None       None      NaN      None     None  ... NaN   

                     geometry  class    Blue   Green     Red     NIR   SWIR1  \
0  POINT (100.58228 13.76398)  urban  4547.0  5006.0  

**เพิ่ม NDVI**

In [50]:
df["NDVI"] = (df["NIR"] - df["Red"]) / (df["NIR"] + df["Red"] + 1e-10)

**แบ่ง Training / Testing**

In [51]:
def create_class(row):

    if row["natural"] == "water":
        return "water"

    elif row["landuse"] in ["industrial","residential","commercial"]:
        return "urban"

    elif pd.notna(row["park_name"]):
        return "green"

    else:
        return None

df["class"] = df.apply(create_class, axis=1)

In [52]:
from sklearn.model_selection import train_test_split

X = df[["Blue","Green","Red","NIR","SWIR1","SWIR2","NDVI"]]
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

**ปรับขนาดข้อมูล + Feature Selection**

In [53]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

**ใช้ Random Forest จำแนกพื้นที่**

In [54]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train_scaled, y_train)

RandomForestClassifier(random_state=42)

**ใช้ Grid Search ปรับพารามิเตอร์**

In [55]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators":[50,100,200],
    "max_depth":[5,10,20]
}

grid = GridSearchCV(
    RandomForestClassifier(),
    param_grid,
    cv=5
)

grid.fit(X_train_scaled, y_train)

best_model = grid.best_estimator_

print(grid.best_params_)

{'max_depth': 5, 'n_estimators': 50}


In [56]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

y_pred = best_model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("Classification Report")
print(classification_report(y_test, y_pred))

Accuracy: 0.6555555555555556
Confusion Matrix
[[ 0  9  7]
 [ 0 30 11]
 [ 0  4 29]]
Classification Report
              precision    recall  f1-score   support

       green       0.00      0.00      0.00        16
       urban       0.70      0.73      0.71        41
       water       0.62      0.88      0.72        33

    accuracy                           0.66        90
   macro avg       0.44      0.54      0.48        90
weighted avg       0.54      0.66      0.59        90



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
